# Implementing a Simple Retrieval-Augmented Generation (RAG)
This lab is based on a workshop created by our friends at the [Swiss-AI-Center](https://www.hes-so.ch/swiss-ai-center).

Original Authors:
- Jonathan Guerne, Research Assistant at Haute Ecole Arc Ingénierie, Switzerland
- Henrique Marques Reis, Research Assistant at Haute Ecole Arc Ingénierie, Switzerland
- Célien Donzé, Scientific Collaborator at HEIA-FR, Switzerland

Adapted by:
- Francesco Carrino, Professor at Haute Ecole d'Ingénierie, Valais/Wallis, Switzerland

## Introduction
This notebook explains how to create a RAG (Retrieval-Augmented Generation) system to answer questions about online or PDF documents. We will use a self-hosted LLM with Ollama to generate answers to the questions. Additionally, we will use a vector database to retrieve relevant documents for answering the questions.

### Documentation and References
Technologies and tools used in this lab:
- [Large Language Model (LLM)](https://en.wikipedia.org/wiki/Large_language_model): a model that generates text (typically) based on the transformer architecture. In this lab, we will use open-source models like LLAMA 3.x.
- [Ollama](https://ollama.com/): Ollama is a platform that democratizes access to LLMs by enabling users to run them locally on their machines. In this lab, it will be used to run the LLMs on your machine or in the cloud (Kaggle).
- [Vector database](https://en.wikipedia.org/wiki/Vector_database): A vector database is a relational database system specifically designed to process vectorized data. Unlike conventional databases that contain information in tables, rows, and columns, vector databases work with vectors—arrays of numerical values that signify points in multidimensional space. When used with text data, vector databases can store and retrieve information based on the semantic meaning of the text, rather than just the text itself. Through a process called embedding, text data is converted into vectors that represent the **meaning** of the text. This means text with similar meaning is stored close to each other in the vector space. We will use [FAISS](https://python.langchain.com/docs/integrations/vectorstores/faiss/), an open-source library for efficient similarity search and clustering of dense vectors. In this lab, it will be used to store information about PDF and web documents.
- [LangChain](https://python.langchain.com/docs/introduction/): LangChain is a framework for developing applications powered by large language models (LLMs). It is based on the idea of "chains," where a chain defines sequences of actions, where each step can involve querying an LLM, prompt management, manipulating data, or interacting with external tools or applications. In this lab, it will be used to put together the LLM and the vector database to create a RAG system.
- [Retrieval-Augmented Generation (RAG)](https://en.wikipedia.org/wiki/Retrieval-augmented_generation): RAG is a technique to improve LLMs by adding a retrieval component from an external data source. This is the main goal of this tutorial.

### Structure and Goals
This lab is divided into four parts. In the first three parts, you will be guided to install and implement a simple RAG working on PDF files. Your goal in this part is to understand the code. We provide some questions to guide your learning through the code.

The last part is a challenge where you will have to use what you have learned to implement a RAG working on web pages.

1. **Installation**: Install the necessary libraries and download the data. We provide two guides: one for running the code on Kaggle (or Google Colab), and another for running the code locally.
2. **Creating and Manipulating Prompts**: We will create prompts and simple queries for the LLM and the vector database.
3. **Creating a RAG System**: We will create a simple RAG system to answer questions about PDF documents. The PDFs will be embedded in the vector database and the LLM will generate the answers.
4. **Challenge**: In this last part, *you* will implement a RAG system to answer questions about web pages.

---

## 1. Installation

If your machine is not very performant, you can use Kaggle or Google Colab to run this notebook. The following code block installs the necessary packages.
If you choose to run the code on Kaggle, login to your account and import this notebook. **It is recommended** to have a verified account to have access to the GPU.

As a model for this lab, we propose llama3.1:8b. This model is a (*relatively*) small version of LLAMA 3.1 (it requires about 5GB on your hard disk), making it available for use on your notebook machine environment. We selected this model because it allows "tool calling," a functionality useful for the second activity of this lab.

If you want to try other models, please refer to https://ollama.com/search to check the updated list of models available on the Ollama model repository.

In [1]:
model_name = "llama3.1:8b"  # or 'llama3.2:1b'

Below you can find the instructions to install the necessary packages either on Kaggle (Section 1.1) or on your local machine (Section 1.2).

### 1.1 Installing Ollama (Kaggle or Google Colab)

If you are in Kaggle or Google Colab, you can install Ollama using the following code block. If want to run this notebook locally, see the next section.

These instructions are based on the Medium post titled [How to Run Ollama Models on Google Colab and Kaggle: A Complete Guide](https://medium.com/@mradul18varshney/how-to-run-ollama-generative-ai-models-on-google-colab-and-kaggle-a-complete-guide-3e01fc12512f).

In [ ]:
# Package installation (ONLY on Kaggle or Google Colab)
!pip install -q langchain langchain-classic langchain-huggingface langchain-community faiss-cpu pymupdf pypdf sentence_transformers rich wget python-dotenv cryptography unstructured accelerate

NOTE: you may receive an error message such as `ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.`. This is a known issue and can be ignored.

#### 1.1.1 Setting up the environment (Kaggle or Google Colab)

This updates the package list and installs curl, which is then used to fetch the Ollama installation script.

In [ ]:
# Update system files and install curl and zstd
!apt-get update && apt-get install -y curl && apt-get install -y zstd

#### 1.1.2 Installing Ollama (Kaggle or Google Colab)

This will install Ollama automatically.

You will probably get some warning messages such as:

- *WARNING: systemd is not running*
- *WARNING: Unable to detect NVIDIA/AMD GPU. Install lspci or lshw to automatically detect and install GPU dependencies*

You can ignore these messages.

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

#### 1.1.3 Starting the Ollama server (Kaggle or Google Colab)

Here, we use Python’s `subprocess.Popen()` to run the Ollama server in the background, listening on the specified host and port (127.0.0.1:11438). The server will be available for communication via the API.

You will need this address later for the RAG model.

In [ ]:
import subprocess
import time
import os

# Set the environment variable for the Ollama host and port
os.environ['OLLAMA_HOST'] = '127.0.0.1:11438'

# Start the Ollama server in a subprocess
ollama_process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Wait for the server to start
print("Starting Ollama server...")
time.sleep(1)  # Wait for the server to initialize

print("Ollama server is running on 127.0.0.1:11438")

#### 1.1.4 Pulling an open-source LLM model (Kaggle or Google Colab)
This command downloads the llama3.2:1b model from Ollama’s servers (a small version of LLAMA 3.2), making it available for use on your notebook machine environment.
Also, refer here https://ollama.com/search for checking the list of models available on Ollama model repository.

In [ ]:
# Pull a model (e.g., llama)
print("Pulling the llama model...")
subprocess.run(['ollama', 'pull', model_name]) # model_name is defined at the beginning of this code block

#### 1.1.4 Interacting directly with the model (Kaggle or Google Colab)

To test the installation and the model, we can use the following code block to interact with the model via HTTP requests.

In [ ]:
import requests
import json

def generate_response(query):
    # Define the API URL for the same host as mentioned when starting the server.
    url = "http://localhost:11438/api/chat"
    
    # Define the payload (data) to send in the POST request
    payload = {
        "model": model_name,
        "messages": [
            {
                "role": "user",
                "content": query
            }
        ],
        "keep_alive": 0,
        "stream": False
    }
    
    headers = {
        "Content-Type": "application/json"
    }
    
    # Make the POST request to the API
    try:
        response = requests.post(url, data=json.dumps(payload), headers=headers)
    
        # Check if the request was successful (status code 200)
        if response.status_code == 200:
            output = response.text
            return output
        else:
            print(f"Failed to get a response. Status code: {response.status_code}")
            return ("Error response:", response.text)
    except Exception as e:
        print(f"An error occurred: {e}")

Once the server is running and the model is pulled, you can start interacting with the model using Python code. In this example, the `generate_response()` function sends a POST request to the Ollama API, passing in the user query. The model processes the query and returns a generated response.

*Note: If you change the model you will get the error that the model is not downloaded and you have to pull it using ollama pull. Also, check the model name to match with the model you pulled using `ollama pull` command.*

In [ ]:
# Using the function to get the output
# The output is in JSON format, and you can parse it using Python’s json module.

query = "What is the difference between AI and ML? Give a short answer."
output = generate_response(query)
json.loads(output)['message']['content']

#### 1.1.5 Using Ollama’s Python SDK (Kaggle or Google Colab)

To interact with the Ollama server, you can use the Python SDK. The SDK is available on PyPI and can be installed using pip. The Ollama python SDK is a wrapper around the Ollama API, which allows you to interact with the server using Python code. The SDK provides a simple and easy-to-use interface for sending requests to the server and receiving responses from the model. This is what we will use in this lab to interact with the model.

In [ ]:
!pip install -q ollama langchain_ollama

The code below interacts with the Ollama model using the Python SDK to send a query and print the generated response.

*Note: (again) If you change the model, you will get an error that the model is not downloaded and you have to pull it using `ollama pull`.*

In [ ]:
from ollama import chat
from ollama import ChatResponse

response: ChatResponse = chat(model=model_name, messages=[
  {
    'role': 'user',
    'content': 'Show me the lyrics of any song?',
  },
])
print(response['message']['content'])

If all works, you should see the generated response from the model. In my test, I got the lyrics from the song "Yesterday" by The Beatles. What did you get?

- - - 

### 1.2 Installing Ollama (locally on your machine)

This guide will help you install Ollama on your local machine. You need a fairly good machine to run LLM models and space on your hard disk to store the LLM models (several GB). If you have a machine with a GPU, you can run the models faster.
However, there are lighter models that can run on CPU and provide results good enough for this lab and many applications (e.g., you can try the "llama3.2:1b" or "qwen2.5:0.5b" model).

#### 1.2.1 Download and install Ollama
Ollama is available for macOS, Linux, and Windows. Go to the official [website](https://ollama.com/) and download the version for your operating system. Follow the instructions to install it.

#### 1.2.2 Running Ollama
After installing Ollama, you should see the Ollama icon in your system tray. You can use it to see the logs and quit the application.

Usually, after the installation, Ollama is *already* running. 

1. [Optional] If you want to run it again, you can run the following command in your terminal:

    ```bash
    ollama serve
    ```

2. To pull a model, go to the [Ollama model list](https://ollama.com/search) and select a model suitable for your machine. Start with a small model. You can, for instance, try the small version of [LLAMA 3.2](https://ollama.com/library/llama3.2).
Follow the instructions for the model you selected. For example, we suggest you try LLAMA 3.2 (the 1B parameters version). To use this version, run in your terminal:

    ```
    ollama run llama3.2:1b
    ```
    The `run` command will pull (download) the model to your machine and then run it, exposing it via the API started with `ollama serve`.

3. [Optional] If you want to *pull* a different model, you can run:

    ```bash
    ollama pull <model_name>
    ```
    The command `ollama list` will show you the list of models available on your machine. To switch between models available on your machine, you can just use the `run` command. The `stop` command will stop a running model.

NOTES:
- If the model is not present on your machine, the `run` command will automatically try to pull it.
- Models take a lot of space on your hard disk. Make sure you have enough space before pulling a model. Remove unnecessary models using the `ollama remove <model_name>` command.

#### 1.2.3 Get Ollama address
In order to interact with the model, you need to know the address where the model is running. Ollama binds to the local address 127.0.0.1 on port 11434 by default.

This means that you can note http://localhost:11434

#### 1.2.4 Package installation (with UV)
We will install all the necessary packages (ollama-related packages included) using UV. We provide a `pyproject.toml` file with the necessary dependencies. We assume you have [UV installed](https://github.com/hei-synd-aml/lab-0-TutoUv) on your machine. Open a terminal in the project location and run the following command to install the packages.

```bash
uv sync
```

---

## 2. Creating a prompt and interacting with the model

OK, now that you have the Ollama server running (either on Kaggle or on your Local machine) and you pulled the model, let's start by creating a prompt and interacting with the LLM model.
In this section, we will create a prompt and interact with the model to generate text.

### Imports

In [2]:
import os
from pathlib import Path

import wget
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.prompts import PromptTemplate
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaLLM
from rich.console import Console
from rich.markdown import Markdown

console = Console()

### Constants

In [3]:
# OLLAMA_ADDRESS = "http://localhost:11438" # use this if you are running Ollama on Kaggle or Colab
OLLAMA_ADDRESS = "http://localhost:11434"  # use this if you are running Ollama on your local machine

model_name = model_name # Defined  replace with the LLM name you pulled from the OLLAMA


### Connecting to LLM

NOTE:
- The first time you run this cell, it may take a while to connect to the model, as the model may need to be loaded into memory.
- if you get "ConnectionError", check if the Ollama server is running and the address is correct.

In [5]:
llm = OllamaLLM(
    model=model_name,
    base_url=OLLAMA_ADDRESS,
    temperature=0.1,  # Will be explained later
    stop=["<end_of_turn>"],
)

llm.generate(["What's your favorite color?"]).generations[0][0].text

"I don't have a personal preference for colors, nor do I have personal experiences or emotions. I'm a large language model, my purpose is to provide information and assist with tasks, but I don't have subjective experiences or opinions. However, I can tell you about different colors, their meanings, and associations in various cultures if you're interested!"

**Question**:
- What is the role of the temperature parameter in a LLM?

**Answer**:

- TO DO

### Creating a prompt

A prompt is generally divided into two parts: the context and the question.

The context provides the information that the model will use to generate its answer, while the question specifies what the model is expected to respond to.
Typically, the context contains a system prompt, which explains the expected behavior from the model, and other additional information that can help the model to provide the right answer. Typically, when used for chatbots, in the context there is a summary (or the entierity!) of all the previous exchange the user had with the model in that conversation. 

LangChain uses templates and markers indicating where to insert the user's question and the context within the template. For more infomation, you can check [Langchain prompt templates documentation](https://python.langchain.com/v0.2/docs/concepts/#prompt-templates).

In [6]:
# A langchain's template is nothing more than a string with {markers} indicating placeholder where other text will be inserted when creating a prompt.

template = """
You are an helpful assistant that answer the question in detail.

Human input: {question}
Assistant:"""

prompt = PromptTemplate(input_variables=["question"], template=template)
prompt

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='\nYou are an helpful assistant that answer the question in detail.\n\nHuman input: {question}\nAssistant:')

### Creating the chain and start a conversation

In [7]:
# Simple chain putting together a prompt and a model 
simple_llm_chain = prompt | llm

In [11]:
result = simple_llm_chain.invoke(input="What does the term RAG refer to?") # we ask a question to the LLM

console.print(Markdown(result)) # we print the output proposed by the LLM

The term "RAG" can have multiple meanings depending on the context. Here are some of the most common               
interpretations:                                                                                                   

 1 Recombinant Activated Gene: In molecular biology, RAG (Recombinase-Activating Genes) refers to a pair of genes  
   that play a crucial role in the development and function of the immune system, particularly in the process of   
   V(D)J recombination. This process is essential for generating diversity in the immune system's B cells and T    
   cells.                                                                                                          
 2 Rapid Airborne Growth: In aviation and meteorology, RAG can refer to conditions that favor rapid growth of      
   clouds or thunderstorms due to the presence of moisture-laden air moving into an area with strong updrafts. This
   term is often used in weather forecasting to predict severe weather events such as thunderstorms.               
 3 Rapid Airborne Growth (aviation training): In aviation, RAG can also refer to a flight simulator or a training  
   aircraft that is used for intensive flying training, particularly for pilots-in-training. The goal of this type 
   of training is to simulate real-world scenarios and conditions in a controlled environment.                     
 4 Reactive Attachment Grief: This term refers to the emotional pain experienced by individuals who have difficulty
   forming healthy attachments due to early life trauma or neglect. It's a concept that has been explored in       
   psychology, particularly in attachment theory.                                                                  
 5 RAG (Radio Amateur Guild): In amateur radio, RAG can refer to a local club or organization of amateur radio     
   operators who come together for social and educational activities related to the hobby.                         
 6 RAG (Racing Association Group): This term is used in various racing contexts, such as horse racing, car racing, 
   or other forms of competitive sports, where it refers to an association or group that oversees and organizes    
   events within a specific category or region.                                                                    
 7 RAG (Random Access Generator): In computing, RAG can refer to software or hardware components responsible for   
   generating random numbers or sequences used in various applications such as simulations, modeling, or           
   encryption.                                                                                                     
 8 RAG (Reactive Attachment Group): This term is sometimes used interchangeably with Reactive Attachment Grief but 
   may also refer to a group of individuals who share similar experiences and are working through attachment issues
   together under the guidance of therapists or support groups.                                                    

The specific meaning of "RAG" would depend on the context in which it's being discussed.

**Questions**:
- What is the relationship between the `input` parameter in `llm_chain.invoke(input="")` and the prompte template defined above?
- Did the model generate a good answer? Why?
- Feel free to change the prompt and the input and see what happens. Try to consider this point: how can you judge the quality of the answer? How could you create a proper evaluation protocol to evaluate the quality of the answers? How is it done in the industry/research field?

**Answers**:
- TO DO




---

## 3. Using the RAG

For now, we have a model that can generate answers to questions. To provide answers, it only uses its knowledge acquired during training and the context we provide. But what if we could provide the model with a set of documents and ask it to retrieve the most relevant ones to answer the question?
That is the idea behind Retrieval-Augmented Generation (i.e., we literally augment the generation of text by enriching the context with information contained in relevant documents).

A typical use case is a company having a huge internal knowledge base (e.g., a collection of documents, web pages, etc.) and wanting to provide its employees with a chatbot that can answer questions about the knowledge base. The chatbot will retrieve the most relevant documents from the knowledge base and use them to generate the answer. This is particularly interesting for companies that have a lot of internal documents that may contain sensitive data.

<img src="./img/RAG_schema.png" alt="Generic schema of a RAG system" width="600"/>

[*(Source of the image)*](https://www.promptingguide.ai/research/rag)

### Exploring pdf files

Here we demonstrate RAG using a collection of *pdf* documents. We will use the scientific articles available online in open access. We will download the documents and store them in the `data` folder. You are free to test the code with your own documents. Just make sure they are in the `data` folder.

NOTE: *pdf* documents are surprisingly complex documents. They can contain text, images, tables, etc. Actually, two different *pdf* documents can have very different structures. Some additional preprocessing may be needed to extract the text from the pdfs. However, for the purpose of this lab, the code provided below should work for most of the documents.

In [12]:
# Create the "data/PDFs" folder if it doesn't exist
PDF_FOLDER = Path("data/PDFs")
os.makedirs(PDF_FOLDER, exist_ok=True)

urls = [
    "https://simg.baai.ac.cn/paperfile/25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf", # Title: "Retrieval-Augmented Generation for Large Language Models: A Survey"
    "https://arxiv.org/pdf/2405.07437.pdf", # Title: "Evaluation of Retrieval-Augmented Generation: A Survey"

]

# Download the PDFs
for url in urls:
    name = url.split("/")[-1]
    if not (PDF_FOLDER / name).is_file():
        filename = wget.download(url, f"data/PDFs/{name}")
console.print("Pdf file downloaded successfully.", style="bold green")

Pdf file downloaded successfully.

### Loading and preprocessing PDF files

In [13]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n', '\n', r'(?<=\. )', r'(?<=\, )', ' ', ''] # the r indicates a raw string
)

***Questions***
- Investigate and explain the role of `chunk_size` and `chunk_overlap` in the code above.
- What are the advantages and disadvantages of having bigger/smaller `chunk_size`?
- What are the advantages and disadvantages of having bigger/smaller `chunk_overlap`?
- What is a difference between a token and a chunk?
- What is the parameter `separators` used for?
- What does the **Recursive** Character Text Splitter does differently compared to a **Simple** character text splitter? 

***Answers***
- TO DO

In [14]:
# All data will be in the list documents
documents=[]

In [15]:
from langchain_classic.document_loaders import DirectoryLoader, PyPDFLoader
# Load and process the text files
# loader = TextLoader('single_text_file.txt')
loader = DirectoryLoader(PDF_FOLDER, glob="./*.pdf", loader_cls=PyPDFLoader)

pdf_docs = loader.load()
len(pdf_docs)

# tokenize pdf
documents.extend(text_splitter.split_documents(pdf_docs))
len(documents)

518

#### What is a "Document" in langchain?
In LangChain, a  [document](https://python.langchain.com/api_reference/core/documents/langchain_core.documents.base.Document.html) is composed of two elements: the text (contained in the field `page_content`) and the metadata (contained in the field `metadata`).

The metadata is a dictionary that can contain any information you want to store about the document. In this case, we store the URL of the website.

In [16]:
# To understand what is going on, we print a document. The text:
print(documents[1].page_content)

Abstract. Retrieval-Augmented Generation (RAG) has recently gained traction
in natural language processing. Numerous studies and real-world applications
are leveraging its ability to enhance generative models through external informa-
tion retrieval. Evaluating these RAG systems, however, poses unique challenges
due to their hybrid structure and reliance on dynamic knowledge sources. To
better understand these challenges, we conduct A Unified Evaluation Process of


In [17]:
# To understand what is going on, we print a document. The text:
print(documents[1].metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-07-04T00:23:56+00:00', 'author': '', 'keywords': '', 'moddate': '2024-07-04T00:23:56+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data\\PDFs\\2405.07437.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1'}


**Question**:
- why is important to also store metadata when doing RAG? Any idea?

**Answer**:
- TO DO

### Embedding a PDF in a vectorstore

The code below creates a vectorstore from the documents. The vectorstore is a database that stores the embeddings of the documents. The embeddings are used to retrieve the most relevant documents for a given question.
It may take a while to create the vectorstore, depending on the number of documents and the size of the documents.

However, if you have new documents, you can just add them to the vectorstore using the `add_documents` method instead of recreating the vectorstore from scratch. 

In [18]:
VECTORSTORES_DIR = Path("data/vectorstores_pdf")

In [19]:
EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5" # locally you can also use smaller models. Check the HuggingFace model hub for more models with different sizes, performance and languages: https://huggingface.co/spaces/mteb/leaderboard

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    # to use the "cuda" configuration, you need an nvidia GPU, and to install 
    # On Kaggle, you set to use as accelerator GPU P100 (you need a verified account)
    # model_kwargs={"device": "cpu"}, # change to cuda -> cpu if you do not have a Nvdia GPU
    model_kwargs={"device": "cpu"}, # change to mps for MacOS M1/M2
    encode_kwargs={"normalize_embeddings": True},
)

The instructions below embeds the documents in the vectorstore. Then, we can save the vectorstore to disk. In this way, we can load the vectorstore and the documents at a later time without having to re-embed the documents. If we get new relevant documents, we can add them to the vectorstore and re-embed them. If interested, you can check the [FAISS documentation](https://python.langchain.com/api_reference/community/vectorstores/langchain_community.vectorstores.faiss.FAISS.html) for more information.

In [20]:
vectorstore = FAISS.from_documents(documents=documents, embedding=embedding_model)

In [21]:
vectorstore.save_local(VECTORSTORES_DIR)

### Loading a vectorstore
We already have the vectorstore with the pdfs embedded in the variabale `vectorstore`. For completeness, in the code above, we show how to load the vectorstore from disk.

In [22]:
vectorstore = FAISS.load_local(
    VECTORSTORES_DIR, embedding_model, allow_dangerous_deserialization=True
)

### New prompt

In RAG we need to add another marker to indicate where the new information (or context) should be inserted for this we use the variable named `{context}`.

In [23]:
prompt = """
Use the following pieces of context to answer the question at the end.
Don't try to make up an answer and only use the information you know.
Use three sentences maximum and keep the answer as concise as possible.
You must answer in english. If you don't know the answer, say "I don't know".
Context:
{context}

Question:
{input}

Answer:
"""

prompt_template = PromptTemplate(input_variables=["context", "input"], template=prompt)
prompt_template

PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nUse the following pieces of context to answer the question at the end.\nDon\'t try to make up an answer and only use the information you know.\nUse three sentences maximum and keep the answer as concise as possible.\nYou must answer in english. If you don\'t know the answer, say "I don\'t know".\nContext:\n{context}\n\nQuestion:\n{input}\n\nAnswer:\n')

***Questions***
- If you look closely, you will notice that the prompt above has 4 regions into which we can insert (or not) information (the system prompt, the context, the input and the answer). What are these regions? Explain the role of each one.

***Answers***
- TO DO

**Tips and Tricks**: You can modify the system prompt to get an answer closer to the one you are looking for.

- Change the language, style, or length.
- Add more information to the prompt, such as suggesting a style or tone for the answer (e.g., "be concise", "be formal", "be informal", "be friendly", etc.).
- Provide more context (e.g., "the text is a scientific paper", "the text is a news article", etc.).
- Specify the question type (e.g., "the question is about the content of the text", "the question is about the author of the text", etc.).
- Define the answer format (e.g., "the answer should be a summary of the text", "the answer should be a list of keywords", etc.).
- Include information about the user (e.g., "the user is a scientist", "the user is a student", etc.).
- Mention details about the model (e.g., "the model is a chatbot", "the model is a search engine", etc.).
- Indicate the desired output format (e.g., "the output should be in JSON format", "the output should be in XML format", etc.).
- Provide specific examples to illustrate the impact of prompt modifications.
- Highlight any limitations or constraints of the model.
- And more...

**Notes and Warnings**:

- Different models may have different markers for the question and context. Check the documentation of the model you are using. This is a true challenge of prompt-engineering.
- Some models (especially smaller ones) may not support different languages.

### Similarity search

<img src=".\img\Similarity_search_schema.png" alt="Generic schema of a RAG system" width="600"/>

[*(Source of the image)*](https://python.langchain.com/docs/concepts/vectorstores/)

In [24]:
# Top k of chunks to retrieve from the vectorstore
NB_RETRIVED_CHUNKS = 8

question_answer_chain = create_stuff_documents_chain(llm=llm, prompt=prompt_template)
retriever = vectorstore.as_retriever(
    search_type="mmr", #  Can be "similarity" (default), "mmr", or "similarity_score_threshold".
    search_kwargs={
        "k": NB_RETRIVED_CHUNKS,
    }
)
chain_with_retriever = create_retrieval_chain(retriever, question_answer_chain)

In the code above, multiple things go on at the same time.
[create_stuff_documents_chain](https://python.langchain.com/docs/versions/migrating_chains/stuff_docs_chain/) combines documents by concatenating them into a single context window. 

**Questions**:
- What is a retriever? 
- What is the meaning of the parameter: "mmr"? What is the difference between "mmr", "similarity" and "similarity_score_threshold"?
- What is the meaning of the parameter "k" and how increasing/decreasing it may impact the results?
- When using "mmr", there are other parameters for the retriever. One of them is `lambda_mult`. What is the meaning of this parameter and what are the possible values? 
- What `create_retrieval_chain` does?

**Answers**:
- TO DO

### "Chatting" with a pdf

Everything is ready to chat with the model: the content of the pdf is embedded in the vectorstore, the model is running, and the chain is created. Let's ask some question to our pdf! Maybe they can help, where the model alone could not.

In [25]:
result = chain_with_retriever.invoke(input={"input": "What does the term RAG refer to?"})

console.print(Markdown(result["answer"]))

The term RAG refers to Retrieval-Augmented Generation, which is a model that combines a pre-trained retriever with 
a pre-trained sequence-to-sequence model. In the context of Large Language Models, RAG specifically refers to a    
model that first retrieves relevant representations when answering questions or generating text.

**Question**:
- What is the size of `result`? What do you expect to find in `result`?
- Let us consider again the "temperature" parameter. For an application that needs to provide accurate information coming from existing document, is it better to use a high or low temperature?
- Suggestion: try to change it and see the difference in the answer.

**Answer**:
- TO DO

### Explaining the output

The `result` dictionary contains much more than the simple answer. Thanks to the metadata, we can see the title of the document, the page number, and the context where the answer was found!

In [26]:
console.print(result)

{
    'input': 'What does the term RAG refer to?',
    'context': [
        Document(
            id='a8120dec-7a16-4854-a499-e3a0cba0b854',
            metadata={
                'producer': 'pdfTeX-1.40.25',
                'creator': 'LaTeX with hyperref',
                'creationdate': '2023-12-19T04:03:26+00:00',
                'author': '',
                'keywords': '',
                'moddate': '2023-12-19T04:03:26+00:00',
                'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea 
version 6.3.5',
                'subject': '',
                'templateversion': 'IJCAI.2024.0',
                'title': '',
                'trapped': '/False',
                'source': 'data\\PDFs\\25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf',
                'total_pages': 26,
                'page': 2,
                'page_label': '3'
            },
            page_content='contents of the survey.\n2 Background\nIn this chapter, we will introduce the definition 
of RAG, as\nwell as the comparison between RAG and other model opti-\nmization techniques, such as 
fine-tuning.\n2.1 Definition\nThe meaning of RAG has expanded in tandem with techno-\nlogical developments. In the 
era of Large Language Mod-\nels, the specific definition of RAG refers to the model, when\nanswering questions or 
generating text, first retrieving rele-'
        ),
        Document(
            id='39e9fe5d-c229-4060-8ade-41be3a43a673',
            metadata={
                'producer': 'pdfTeX-1.40.25',
                'creator': 'LaTeX with hyperref',
                'creationdate': '2024-07-04T00:23:56+00:00',
                'author': '',
                'keywords': '',
                'moddate': '2024-07-04T00:23:56+00:00',
                'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea 
version 6.3.5',
                'subject': '',
                'title': '',
                'trapped': '/False',
                'source': 'data\\PDFs\\2405.07437.pdf',
                'total_pages': 21,
                'page': 8,
                'page_label': '9'
            },
            page_content='Navigating the intricate terrain of evaluating RAG systems necessitates a nuanced 
un-\nderstanding of the metrics that can precisely quantify the evaluation targets. However,\ncreating evaluative 
criteria that align with human preferences and address practical con-\nsiderations is challenging. Each component 
within the RAG systems requires a tailored\nevaluative approach that reflects its distinct functionalities and 
objectives.'
        ),
        Document(
            id='9691d5ec-951c-428e-94a6-ac8edf789e75',
            metadata={
                'producer': 'pdfTeX-1.40.25',
                'creator': 'LaTeX with hyperref',
                'creationdate': '2023-12-19T04:03:26+00:00',
                'author': '',
                'keywords': '',
                'moddate': '2023-12-19T04:03:26+00:00',
                'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea 
version 6.3.5',
                'subject': '',
                'templateversion': 'IJCAI.2024.0',
                'title': '',
                'trapped': '/False',
                'source': 'data\\PDFs\\25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf',
                'total_pages': 26,
                'page': 9,
                'page_label': '10'
            },
            page_content='representations?\nIn RAG, semantic space is the multidimensional space where\nquery and 
Document are mapped. When we perform re-\ntrieval, it is measured within the semantic space. If the se-\nmantic 
expression is not accurate, then its effect on RAG is\nfatal, this section will introduce two methods to help us 
build\na accurate semantic space.\nChunk optimization\nWhen processing external documents, the first step is 
chunk-\ning to obtain fine-grained features. Then the chunks are Em-

**Questions**:
- The `result`dictionary contains thee keys: `input`, `context`, and `answer`. What do they contain? Do they remember you of something?
- Let us focus on the `context` key. It contains a list of documents. Each document is a dictionary with three keys: `id`, `metadata`, and `page_content`. What do they contain? What is the relationship between what you see and the "chunk" you saw before?
- Which of this information is used by the retriever to retrieve the documents?
- Which of this information is used by the LLM to generate the answer?

**Answers**:
- TO DO

### Add filters using metadata
In the code above, we did not yet use information in metadata. Below, we show how to filter the documents based on just that. For instance, what if we want to limit (filter) the retrieval to documents with a specific name?

In [ ]:
# Top k of chunks to retrieve from the vectorstore
NB_RETRIVED_CHUNKS = 8

question_answer_chain = create_stuff_documents_chain(llm=llm, prompt=prompt_template)
retriever = vectorstore.as_retriever(
    search_type="mmr", #  Can be "similarity" (default), "mmr", or "similarity_score_threshold".
    search_kwargs={
        "k": NB_RETRIVED_CHUNKS,
        
        # this below is the only new part
        "filter": { 
            "source": "data\\PDFs\\25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf", # Filter by source
        }
    }
)
chain_with_retriever = create_retrieval_chain(retriever, question_answer_chain)

In [29]:
result = chain_with_retriever.invoke(input={"input": "What does the term RAG refer to?"})

console.print(Markdown(result["answer"]))

The term RAG refers to Retrieval-Augmented Generation, a model that combines a pre-trained retriever with a        
pre-trained seq2seq model (generator). It was first introduced by Lewis et al. in 2020. In the context of Large    
Language Models, RAG specifically refers to the process of retrieving relevant representations when answering      
questions or generating text.

In [30]:
console.print(result)

{
    'input': 'What does the term RAG refer to?',
    'context': [
        Document(
            id='a8120dec-7a16-4854-a499-e3a0cba0b854',
            metadata={
                'producer': 'pdfTeX-1.40.25',
                'creator': 'LaTeX with hyperref',
                'creationdate': '2023-12-19T04:03:26+00:00',
                'author': '',
                'keywords': '',
                'moddate': '2023-12-19T04:03:26+00:00',
                'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea 
version 6.3.5',
                'subject': '',
                'templateversion': 'IJCAI.2024.0',
                'title': '',
                'trapped': '/False',
                'source': 'data\\PDFs\\25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf',
                'total_pages': 26,
                'page': 2,
                'page_label': '3'
            },
            page_content='contents of the survey.\n2 Background\nIn this chapter, we will introduce the definition 
of RAG, as\nwell as the comparison between RAG and other model opti-\nmization techniques, such as 
fine-tuning.\n2.1 Definition\nThe meaning of RAG has expanded in tandem with techno-\nlogical developments. In the 
era of Large Language Mod-\nels, the specific definition of RAG refers to the model, when\nanswering questions or 
generating text, first retrieving rele-'
        ),
        Document(
            id='9691d5ec-951c-428e-94a6-ac8edf789e75',
            metadata={
                'producer': 'pdfTeX-1.40.25',
                'creator': 'LaTeX with hyperref',
                'creationdate': '2023-12-19T04:03:26+00:00',
                'author': '',
                'keywords': '',
                'moddate': '2023-12-19T04:03:26+00:00',
                'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea 
version 6.3.5',
                'subject': '',
                'templateversion': 'IJCAI.2024.0',
                'title': '',
                'trapped': '/False',
                'source': 'data\\PDFs\\25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf',
                'total_pages': 26,
                'page': 9,
                'page_label': '10'
            },
            page_content='representations?\nIn RAG, semantic space is the multidimensional space where\nquery and 
Document are mapped. When we perform re-\ntrieval, it is measured within the semantic space. If the se-\nmantic 
expression is not accurate, then its effect on RAG is\nfatal, this section will introduce two methods to help us 
build\na accurate semantic space.\nChunk optimization\nWhen processing external documents, the first step is 
chunk-\ning to obtain fine-grained features. Then the chunks are Em-'
        ),
        Document(
            id='9e338900-2409-4cec-8166-6f6ccc006c25',
            metadata={
                'producer': 'pdfTeX-1.40.25',
                'creator': 'LaTeX with hyperref',
                'creationdate': '2023-12-19T04:03:26+00:00',
                'author': '',
                'keywords': '',
                'moddate': '2023-12-19T04:03:26+00:00',
                'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea 
version 6.3.5',
                'subject': '',
                'templateversion': 'IJCAI.2024.0',
                'title': '',
                'trapped': '/False',
                'source': 'data\\PDFs\\25a43194-c74c-4cd3-b60f-0a1f27f8b8af.pdf',
                'total_pages': 26,
                'page': 9,
                'page_label': '10'
            },
            page_content='Retrieve-Read flow. Self-RAG [Asai et al., 2023b] fol-\nlows the 
decide-retrieve-reflect-read process, introduc-\ning a module for active judgment. This adaptive and\ndiverse 
approach allows for the dynamic organization of\nmodules within the Modular RAG framework.\n4 Retriever\nIn the 
context of RAG, the ”R” stands for retrieval, serving\nthe role in t

**Questions**:
- Was the answer generated by the model different? 
- Did you notice that the answer was more or less more precise? Why?

**Answers**:
- TO DO

---

## 4. Challenge: getting information from a website

Now, it's your turn. Use the code above as inspiration to:
- interact with a website (instead of a PDF)
- create a vectorstore with the information from the website
- save the vectorstore to disk
- add embeddings from another website to the vectorstore (without recreating it from scratch)
- improve the prompt template to get better answers
- chat with the model
- evaluate the output (with and without RAG)

You can choose any website you want. If do not have any inspiration, we suggest to use https://docs.realgames.co/homeio/en/.
You may ask the model to provide you with a list the available detectors in Home I/O. And try to use filters and metadata to improve the results (maybe pointing the actual webpage containing the correct answer...).


Hint: If you need some help to get started check [this](https://python.langchain.com/docs/how_to/document_loader_web/).

In [ ]:
# TODO:

## Conclusion

In this lab, we learned how to create a simple RAG system to answer questions about PDF documents and websites. We used a vector database to store information about the PDF documents and a self-hosted LLM to generate the answers. We also learned how to create prompts and interact with the model to generate text.

PDF and web documents are just examples of the many applications of RAG systems. RAG can work with any type of document or data source, such as databases, APIs, YouTube videos, Excel files, etc.